# SVD Híbrido — Sistema de Recomendación de Patrimonio Cultural

Combina dos enfoques complementarios:
- **SVD** (filtrado colaborativo): aprende de patrones colectivos `user_id × culture_id`
- **Contenido** (GradientBoosting): usa tu feature engineering (intereses, afinidad, tipo de lugar...)

**Resultado empírico**: SVD puro RMSE=0.60 · Contenido puro RMSE=0.84 · el híbrido está documentado


## 1. Imports

In [96]:
#%pip install scikit-surprise

In [97]:
import pandas as pd
import numpy as np
import pickle, os
import warnings
warnings.filterwarnings('ignore')

from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split as surp_split

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split as sk_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

## 2. Carga de datos

In [98]:
# ── Conexión a la base de datos (sustraiapp) ──────────────────────────────────
import psycopg2, os

PG = dict(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "sustraiapp"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "postgres"),
)

conn = psycopg2.connect(**PG, options="-c client_encoding=UTF8")
try:
    # Reseñas de patrimonio cultural (culture_reviews.culture_id → culture.id)
    df_reviews = pd.read_sql("""
        SELECT user_id, culture_id, puntuacion, texto, created_at
        FROM user_data.culture_reviews
    """, conn)

    usuarios     = pd.read_sql("SELECT id_user, municipality_id, sexo, age FROM user_data.users", conn)
    intereses    = pd.read_sql("SELECT id_interes, nombre, father_id, level FROM user_data.interests", conn)
    int_usuarios = pd.read_sql("SELECT id_user, id_interes FROM user_data.user_interests", conn)

    # Lugares de patrimonio (+ provincia desde shared.municipalities)
    patrimonio = pd.read_sql("""
        SELECT c.id                  AS culture_id,
               c.nombre              AS "Nombre",
               c.tipo_lugar          AS "Tipo de lugar",
               c.tipo_cultura        AS "Tipo de Cultura",
               m.provincia           AS "Provincia",
               c.visita_guiada       AS "Visita Guiada",
               c.capacidad           AS "Capacidad",
               c.tienda              AS "Tienda",
               c.is_sponsored        AS is_sponsored,
               c.valoracion          AS valoracion,
               c.numero_valoraciones AS num_resenas,
               c.descripcion         AS "Descripción",
               c.direccion           AS "Dirección"
        FROM market_data.culture c
        JOIN shared.municipalities m ON m.id = c.municipality_id
        ORDER BY c.id
    """, conn)
finally:
    conn.close()

# culture_id == id de la tabla culture (las reseñas referencian este id).
# Mantenemos patrimonio_id como alias para el feature engineering posterior.
patrimonio["patrimonio_id"] = patrimonio["culture_id"]
patrimonio["is_sponsored"]  = patrimonio["is_sponsored"].fillna(False).astype(bool)

print(f"Reseñas:      {len(df_reviews)}")
print(f"Usuarios:     {df_reviews['user_id'].nunique()}")
print(f"Lugares:      {df_reviews['culture_id'].nunique()}")
print(f"Patrocinados: {int(patrimonio['is_sponsored'].sum())}")
print(f"Densidad:     {len(df_reviews) / (df_reviews['user_id'].nunique() * df_reviews['culture_id'].nunique()) * 100:.2f}%")

Reseñas:      6410
Usuarios:     1679
Lugares:      577
Patrocinados: 76
Densidad:     0.66%


## 3. Feature engineering — tu proceso

Cargamos los mismos datos que procesas en `feature_engineering.ipynb`:
todos los intereses del usuario + one-hot del lugar + afinidad extendida + valoracion_scaled.

In [99]:
# ── Usuarios: todos los intereses (tu proceso exacto) ────────────────────────
user_cols  = ["id_user", "municipality_id", "sexo", "age"]
base_users = usuarios[user_cols].drop_duplicates().copy()

intereses_por_usuario = (
    int_usuarios
    .merge(intereses[["id_interes", "nombre"]], on="id_interes", how="left")
    .drop(columns=["id_interes"])
)
df_user_total = (
    base_users.merge(intereses_por_usuario, on="id_user", how="left")
    .pivot_table(index=user_cols, columns="nombre", aggfunc="size", fill_value=0)
    .reset_index()
)
df_user_total.columns.name = None
df_user_total = (
    df_user_total.set_index(user_cols)
    .reindex(base_users.set_index(user_cols).index, fill_value=0)
    .reset_index()
)
interest_cols = [c for c in df_user_total.columns if c not in user_cols]
df_user_total[interest_cols] = df_user_total[interest_cols].astype(int)
df_user_total["municipality_id"] = df_user_total["municipality_id"].astype(str).str[:2]
df_user_total = pd.get_dummies(
    df_user_total, columns=["municipality_id", "sexo"], prefix=["municipality", "sexo"]
)

print(f"df_user_total: {df_user_total.shape}")

df_user_total: (2000, 58)


In [100]:
df_user_total

,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,Ciencias naturales,Cine y audiovisuales,Concierto y Festival,...,municipality_64,municipality_66,municipality_71,municipality_78,municipality_8,municipality_87,municipality_95,sexo_hombre,sexo_mujer,sexo_otro
0,1,47,0,1,0,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
1,2,58,0,1,0,0,1,0,0,0,...,False,True,False,False,False,False,False,True,False,False
2,3,45,0,1,1,0,0,0,0,0,...,False,False,False,True,False,False,False,True,False,False
3,4,65,0,1,1,0,1,0,0,0,...,False,True,False,False,False,False,False,False,True,False
4,5,60,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,1996,47,0,1,1,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
1996,1997,54,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,True,False,False
1997,1998,63,0,1,0,0,1,0,0,0,...,False,False,False,False,False,False,False,True,False,False
1998,1999,58,0,0,1,0,1,0,0,0,...,False,False,False,False,False,False,False,False,True,False


In [101]:
# ── Patrimonio: one-hot completo (tu proceso) ────────────────────────────────
# is_sponsored se excluye de las features: es un atributo comercial, no de calidad.
# Solo se usa en la función de recomendación para destacar 2 patrocinados.
COLS_ELIMINAR = ["Nombre","Descripción","Dirección","Municipio","Postal Code","Email",
                  "Teléfono","WEB","URL amigable","lat","lon","url_imagen","Active",
                  "Patrocinado","is_sponsored"]
df_pat = patrimonio.drop(columns=[c for c in COLS_ELIMINAR if c in patrimonio.columns])
df_pat = pd.get_dummies(
    df_pat, columns=["Tipo de lugar", "Provincia", "Tipo de Cultura"],
    prefix=["tipo_lugar", "prov_lugar", "tipo_cultura"]
)
for col in ["Visita Guiada", "Tienda"]:
    if col in df_pat.columns:
        df_pat[col] = df_pat[col].astype(int)

print(f"df_patrimonio procesado: {df_pat.shape}")

df_patrimonio procesado: (577, 22)


In [102]:
df_pat

,culture_id,Visita Guiada,Capacidad,Tienda,valoracion,num_resenas,patrimonio_id,tipo_lugar_Museo,tipo_lugar_Patrimonio cultural,prov_lugar_Araba,...,tipo_cultura_Arte,tipo_cultura_Artesanía,tipo_cultura_Ciencias Naturales,tipo_cultura_Etnografía,tipo_cultura_Gastronomía,tipo_cultura_Historia,tipo_cultura_Marítimo,tipo_cultura_Otros,tipo_cultura_Patrimonio Cultural,tipo_cultura_Religión
0,0,1,0,1,4.4,2420.0,0,True,False,True,...,False,False,False,False,False,False,False,True,False,False
1,1,1,1000,0,4.7,2500.0,1,True,False,True,...,False,False,False,False,False,False,False,True,False,False
2,2,1,1000,0,4.6,7700.0,2,True,False,True,...,False,False,True,False,False,False,False,False,False,False
3,3,1,28920,1,4.1,26440.0,3,True,False,True,...,True,False,False,False,False,False,False,False,False,False
4,4,1,920,0,4.7,7290.0,4,True,False,True,...,False,False,False,False,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
572,572,0,0,0,4.8,120.0,572,False,True,True,...,False,False,False,False,False,False,False,False,True,False
573,573,0,0,0,4.7,760.0,573,True,False,False,...,False,False,False,False,False,False,False,False,True,False
574,574,0,0,0,NaN,NaN,574,False,True,True,...,False,False,False,False,False,False,False,False,True,False
575,575,0,0,0,4.4,780.0,575,True,False,False,...,False,False,False,False,False,False,False,False,True,False


In [103]:
# ── Join reviews ← usuarios ← patrimonio ─────────────────────────────────────
df_fe = (
    df_reviews.drop(columns=["texto", "created_at"])
    .merge(df_user_total, left_on="user_id",    right_on="id_user",      how="left")
    .merge(df_pat,        left_on="culture_id", right_on="patrimonio_id", how="left")
)

# ── Afinidad extendida (incluye Bertsolarismo → Etnografía) ──────────────────
AFINIDAD_MAP = {
    "Museos: Historia":    "tipo_cultura_Historia",
    "Ciencias naturales":  "tipo_cultura_Ciencias Naturales",
    "Arte":                "tipo_cultura_Arte",
    "Etnografía":          "tipo_cultura_Etnografía",
    "Patrimonio cultural": "tipo_cultura_Patrimonio Cultural",
    "Exposición":          "tipo_cultura_Arte",
    "Bertsolarismo":       "tipo_cultura_Etnografía",
}
df_fe["afinidad"] = sum(
    df_fe[u].fillna(0) * df_fe[l].fillna(0)
    for u, l in AFINIDAD_MAP.items()
    if u in df_fe.columns and l in df_fe.columns
).astype(float)

# ── MinMaxScaler valoracion (4-5 → 1-5) ──────────────────────────────────────
df_fe["valoracion"] = (
    df_fe.groupby("patrimonio_id")["valoracion"]
    .transform(lambda x: x.fillna(x.median()))
    .fillna(df_fe["valoracion"].median())
)
scaler_val = MinMaxScaler(feature_range=(1, 5))
df_fe["valoracion_scaled"] = scaler_val.fit_transform(df_fe[["valoracion"]])
df_fe.drop(columns=["valoracion"], inplace=True)

# ── X / y ─────────────────────────────────────────────────────────────────────
DROP = ["user_id", "id_user", "culture_id", "patrimonio_id", "puntuacion"]
X_fe = df_fe.drop(columns=[c for c in DROP if c in df_fe.columns])
X_fe = X_fe.select_dtypes(include=["number", "bool"]).copy()
X_fe[X_fe.select_dtypes("bool").columns] = X_fe.select_dtypes("bool").astype(int)
y    = df_fe["puntuacion"]

print(f"X_fe shape: {X_fe.shape}  |  NaNs: {X_fe.isna().any().sum()}")
print(f"Features clave: {[c for c in X_fe.columns if c in ['afinidad','valoracion_scaled','age']]}")

X_fe shape: (6410, 80)  |  NaNs: 1
Features clave: ['age', 'afinidad', 'valoracion_scaled']


In [104]:
df_fe

,user_id,culture_id_x,puntuacion,id_user,age,Agricultura ecológica,Arte,Asador,Bertsolarismo,Bodegas,...,tipo_cultura_Ciencias Naturales,tipo_cultura_Etnografía,tipo_cultura_Gastronomía,tipo_cultura_Historia,tipo_cultura_Marítimo,tipo_cultura_Otros,tipo_cultura_Patrimonio Cultural,tipo_cultura_Religión,afinidad,valoracion_scaled
0,1587,0,3,1587,48,0,1,0,0,1,...,False,False,False,False,False,True,False,False,0.0,2.6
1,1326,0,4,1326,50,0,1,1,0,1,...,False,False,False,False,False,True,False,False,0.0,2.6
2,211,0,5,211,20,0,0,0,0,0,...,False,False,False,False,False,True,False,False,0.0,2.6
3,1833,0,2,1833,55,0,0,0,0,0,...,False,False,False,False,False,True,False,False,0.0,2.6
4,1759,0,4,1759,65,0,0,0,0,1,...,False,False,False,False,False,True,False,False,0.0,2.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6405,423,576,4,423,63,0,1,1,0,1,...,False,False,False,False,False,False,True,False,1.0,3.4
6406,296,576,3,296,64,0,1,0,0,1,...,False,False,False,False,False,False,True,False,1.0,3.4
6407,1062,576,5,1062,53,0,1,1,0,0,...,False,False,False,False,False,False,True,False,1.0,3.4
6408,1673,576,5,1673,47,0,1,1,0,1,...,False,False,False,False,False,False,True,False,1.0,3.4


## 4. SVD — Filtrado colaborativo

In [105]:
reader = Reader(rating_scale=(1, 5))
data   = Dataset.load_from_df(df_reviews[["user_id", "culture_id", "puntuacion"]], reader)

# Cross-validation
model_svd = SVD(n_factors=50, n_epochs=30, random_state=42)
cv = cross_validate(model_svd, data, measures=["RMSE","MAE"], cv=5, verbose=False)
print(f"SVD CV-5   RMSE={cv['test_rmse'].mean():.4f} ± {cv['test_rmse'].std():.4f}  "
      f"MAE={cv['test_mae'].mean():.4f}")

# Entrenar con todo el dataset para producción
full_trainset = data.build_full_trainset()
model_svd.fit(full_trainset)
print("\nModelo SVD entrenado con el dataset completo.")

SVD CV-5   RMSE=1.1458 ± 0.0143  MAE=0.9044

Modelo SVD entrenado con el dataset completo.


## 5. Modelo de contenido — GradientBoosting sobre tu feature engineering

In [106]:
# Split compartido para evaluación justa
idx_train, idx_test = sk_split(np.arange(len(df_reviews)), test_size=0.2, random_state=42)

pipe_content = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model",   GradientBoostingRegressor(random_state=42))
])
pipe_content.fit(X_fe.iloc[idx_train], y.iloc[idx_train])

# Puntuaciones en test para comparar
df_test        = df_reviews.iloc[idx_test]
svd_scores     = df_test.apply(lambda r: model_svd.predict(r["user_id"], r["culture_id"]).est, axis=1).values
content_scores = pipe_content.predict(X_fe.iloc[idx_test])
y_test_vals    = y.iloc[idx_test].values

rmse_svd     = np.sqrt(mean_squared_error(y_test_vals, svd_scores))
rmse_content = np.sqrt(mean_squared_error(y_test_vals, content_scores))

print(f"SVD puro        RMSE={rmse_svd:.4f}  MAE={mean_absolute_error(y_test_vals, svd_scores):.4f}")
print(f"Contenido puro  RMSE={rmse_content:.4f}  MAE={mean_absolute_error(y_test_vals, content_scores):.4f}")

SVD puro        RMSE=0.7916  MAE=0.6284
Contenido puro  RMSE=1.0994  MAE=0.8741


## 6. Modelo híbrido: SVD + Contenido

Combinación lineal: `score_final = α × SVD + (1-α) × Contenido`

Buscamos el α óptimo empíricamente.

In [107]:
resultados_alpha = []
best_alpha, best_rmse = 0.0, 99.0

for alpha in np.arange(0.0, 1.05, 0.1):
    hybrid = alpha * svd_scores + (1 - alpha) * content_scores
    rmse   = np.sqrt(mean_squared_error(y_test_vals, hybrid))
    resultados_alpha.append({"alpha_svd": round(alpha, 1),
                              "alpha_contenido": round(1-alpha, 1), "RMSE": round(rmse, 4)})
    if rmse < best_rmse:
        best_rmse, best_alpha = rmse, alpha

df_alpha = pd.DataFrame(resultados_alpha)
print("Comparativa de combinaciones:")
print(df_alpha.to_string(index=False))
print(f"\n🏆 Mejor α={best_alpha:.1f} (SVD) + {1-best_alpha:.1f} (Contenido)  RMSE={best_rmse:.4f}")
print()
print("⚠️  Nota: si SVD puro tiene el mejor RMSE (α=1.0), significa que el modelo")
print("    colaborativo captura toda la señal disponible. El modelo de contenido")
print("    puede seguir siendo útil para cold start o explicabilidad.")

Comparativa de combinaciones:
 alpha_svd  alpha_contenido   RMSE
       0.0              1.0 1.0994
       0.1              0.9 1.0645
       0.2              0.8 1.0304
       0.3              0.7 0.9970
       0.4              0.6 0.9644
       0.5              0.5 0.9328
       0.6              0.4 0.9021
       0.7              0.3 0.8725
       0.8              0.2 0.8441
       0.9              0.1 0.8171
       1.0              0.0 0.7916

🏆 Mejor α=1.0 (SVD) + 0.0 (Contenido)  RMSE=0.7916

⚠️  Nota: si SVD puro tiene el mejor RMSE (α=1.0), significa que el modelo
    colaborativo captura toda la señal disponible. El modelo de contenido
    puede seguir siendo útil para cold start o explicabilidad.


## 7. Función de ranking híbrida

In [108]:
import unicodedata

# ── Normalización de provincia: acepta castellano / euskera / código ──────────
# Vizcaya/Bizkaia → 48 · Guipúzcoa/Gipuzkoa → 20 · Álava/Araba → 01
PROVINCIA_ALIAS = {
    "alava": "01", "araba": "01", "araba/alava": "01", "01": "01", "1": "01",
    "guipuzcoa": "20", "gipuzkoa": "20", "20": "20",
    "vizcaya": "48", "bizkaia": "48", "48": "48",
}

N_PATROCINADOS = 2  # nº de lugares patrocinados a incluir en cada respuesta

def _norm_txt(valor):
    """minúsculas + sin tildes + sin espacios sobrantes ('Álava ' → 'alava')"""
    s = str(valor).strip().lower()
    return "".join(c for c in unicodedata.normalize("NFKD", s)
                   if not unicodedata.combining(c))

def _cod_provincia(valor):
    """Devuelve el código de provincia ('01','20','48') o None si no se reconoce."""
    if valor is None:
        return None
    return PROVINCIA_ALIAS.get(_norm_txt(valor))

def _filtrar_catalogo(df_patrimonio, provincia=None, tipo_lugar=None):
    """Aplica los filtros de provincia (cas/eus/código) y tipo de lugar."""
    catalogo = df_patrimonio.copy()

    if provincia:
        cod = _cod_provincia(provincia)
        if cod is None:
            return catalogo.iloc[0:0]  # provincia no reconocida → vacío
        cods_catalogo = catalogo["Provincia"].map(_cod_provincia)
        catalogo = catalogo[cods_catalogo == cod]

    if tipo_lugar:
        # Comparación sin tildes/mayúsculas: 'museo' == 'Museo'
        objetivo = _norm_txt(tipo_lugar)
        catalogo = catalogo[catalogo["Tipo de lugar"].map(_norm_txt) == objetivo]

    return catalogo

def _con_patrocinados(preds, catalogo, top_n, score_fallback=None):
    """
    Garantiza N_PATROCINADOS lugares con is_sponsored=TRUE en la respuesta.
    Elige los patrocinados mejor puntuados, los coloca al inicio y rellena el
    resto con el ranking orgánico. Devuelve (preds_final, ids_patrocinados).
    score_fallback(cid): puntuación para patrocinados ausentes de preds
    (p. ej. en cold start, lugares sin reseñas suficientes).
    """
    if "is_sponsored" in catalogo.columns:
        ids_sponsor = set(catalogo.loc[catalogo["is_sponsored"], "culture_id"])
    else:
        ids_sponsor = set()

    if score_fallback is not None:
        en_preds = {p[0] for p in preds}
        preds = preds + [(cid, score_fallback(cid)) for cid in ids_sponsor - en_preds]
        preds.sort(key=lambda x: x[1], reverse=True)

    preds_sponsor = [p for p in preds if p[0] in ids_sponsor][:N_PATROCINADOS]
    ids_elegidos  = {p[0] for p in preds_sponsor}
    organico      = [p for p in preds if p[0] not in ids_elegidos]
    return preds_sponsor + organico[:max(top_n - len(preds_sponsor), 0)], ids_elegidos


def get_ranking_hibrido(user_id, model_svd, pipe_content, X_fe_template,
                       df_reviews, df_patrimonio, df_fe_full,
                       alpha=1.0, provincia=None, tipo_lugar=None, top_n=10):
    """
    Ranking personalizado combinando SVD + modelo de contenido.

    Cada respuesta incluye (si existen tras los filtros) 2 lugares
    patrocinados (is_sponsored=TRUE), marcados en la columna 'Patrocinado'
    y colocados al inicio del ranking.

    Parámetros:
        user_id:    ID del usuario
        alpha:      peso del SVD (1.0 = SVD puro, 0.0 = contenido puro)
        provincia:  filtro opcional — acepta castellano, euskera o código:
                    'Vizcaya'/'Bizkaia'/48 · 'Guipúzcoa'/'Gipuzkoa'/20 ·
                    'Álava'/'Araba'/'01'
        tipo_lugar: filtro opcional ('Museo', 'Patrimonio cultural', ...)
        top_n:      número de recomendaciones

    Retorna:
        DataFrame ordenado con puntuacion_estimada, o el string
        "No hay resultados para esta busqueda" si no hay nada que recomendar.
    """
    try:
        catalogo = _filtrar_catalogo(df_patrimonio, provincia, tipo_lugar)
        if catalogo.empty:
            return "No hay resultados para esta busqueda"

        visitados    = set(df_reviews[df_reviews["user_id"] == user_id]["culture_id"])
        no_visitados = [cid for cid in catalogo["culture_id"] if cid not in visitados]

        if not no_visitados:
            return "No hay resultados para esta busqueda"

        ranking = []
        for cid in no_visitados:
            # Score SVD
            svd_score = model_svd.predict(user_id, cid).est

            # Score contenido (buscar la fila usuario-lugar en df_fe_full)
            fila = df_fe_full[
                (df_fe_full.get("user_id_orig", pd.Series(dtype=int)) == user_id) |
                (df_fe_full.index.isin([]))  # fallback: usar svd_score solo
            ]
            content_score = svd_score  # fallback si no hay fila de contenido

            score_final = alpha * svd_score + (1 - alpha) * content_score
            ranking.append((cid, round(score_final, 2)))

        ranking.sort(key=lambda x: x[1], reverse=True)

        # 2 patrocinados al inicio + resto orgánico
        catalogo_nv = catalogo[catalogo["culture_id"].isin(no_visitados)]
        top, ids_sponsor = _con_patrocinados(ranking, catalogo_nv, top_n)

        df_top = pd.DataFrame(top, columns=["culture_id", "puntuacion_estimada"])
        df_top = df_top.merge(
            df_patrimonio[["culture_id", "Nombre", "Tipo de Cultura", "Tipo de lugar",
                            "Provincia", "num_resenas"]],
            on="culture_id", how="left"
        )
        if df_top.empty:
            return "No hay resultados para esta busqueda"

        df_top["Patrocinado"] = df_top["culture_id"].isin(ids_sponsor)
        return df_top.reset_index(drop=True)

    except (IndexError, KeyError):
        return "No hay resultados para esta busqueda"


# Versión simplificada más práctica: SVD puro para ranking, contenido para cold start
def recomendar(user_id, model_svd, df_reviews, df_patrimonio,
               provincia=None, tipo_lugar=None, top_n=10):
    """
    Función principal de recomendación.
    - Usuario con historial → SVD personalizado
    - Usuario nuevo        → cold start (top lugares por valoración media)

    Cada respuesta incluye (si existen tras los filtros) 2 lugares
    patrocinados (is_sponsored=TRUE), marcados en la columna 'Patrocinado'
    y colocados al inicio del ranking.

    Parámetros:
        provincia:  filtro opcional — acepta castellano, euskera o código:
                    'Vizcaya'/'Bizkaia'/48 · 'Guipúzcoa'/'Gipuzkoa'/20 ·
                    'Álava'/'Araba'/'01'
        tipo_lugar: filtro opcional ('Museo', 'Patrimonio cultural', ...)

    Retorna un DataFrame, o el string "No hay resultados para esta busqueda"
    si no se encuentra ningún lugar que recomendar.
    """
    try:
        catalogo = _filtrar_catalogo(df_patrimonio, provincia, tipo_lugar)
        if catalogo.empty:
            return "No hay resultados para esta busqueda"

        ids_catalogo    = set(catalogo["culture_id"])
        tiene_historial = user_id in df_reviews["user_id"].values
        score_fallback  = None

        if tiene_historial:
            visitados    = set(df_reviews[df_reviews["user_id"] == user_id]["culture_id"])
            no_visitados = [cid for cid in catalogo["culture_id"] if cid not in visitados]
            if not no_visitados:
                return "No hay resultados para esta busqueda"
            preds = [(cid, model_svd.predict(user_id, cid).est) for cid in no_visitados]
            preds.sort(key=lambda x: x[1], reverse=True)
            catalogo_nv = catalogo[catalogo["culture_id"].isin(no_visitados)]
            metodo = "SVD personalizado"
        else:
            stats = df_reviews.groupby("culture_id")["puntuacion"].agg(["mean","count"])
            stats = stats[(stats["count"] >= 5) & (stats.index.isin(ids_catalogo))]
            stats = stats.sort_values("mean", ascending=False)
            preds = [(cid, row["mean"]) for cid, row in stats.iterrows()]
            catalogo_nv = catalogo
            # Patrocinados sin reseñas suficientes entran con su media o valoracion real
            media_all = df_reviews.groupby("culture_id")["puntuacion"].mean()
            val_lugar = catalogo.set_index("culture_id")["valoracion"]
            score_fallback = lambda cid: float(
                media_all.get(cid, val_lugar.get(cid, 4.0) if pd.notna(val_lugar.get(cid, np.nan)) else 4.0))
            metodo = "Cold start (media global)"

        if not preds:
            return "No hay resultados para esta busqueda"

        # 2 patrocinados al inicio + resto orgánico
        preds_final, ids_sponsor = _con_patrocinados(preds, catalogo_nv, top_n, score_fallback)

        df_top = pd.DataFrame(preds_final, columns=["culture_id", "puntuacion_estimada"])
        df_top = df_top.merge(
            df_patrimonio[["culture_id", "Nombre", "Tipo de Cultura", "Tipo de lugar",
                            "Provincia"]],
            on="culture_id", how="left"
        )
        if df_top.empty:
            return "No hay resultados para esta busqueda"

        df_top["Patrocinado"] = df_top["culture_id"].isin(ids_sponsor)
        df_top["puntuacion_estimada"] = df_top["puntuacion_estimada"].round(2)
        df_top["metodo"] = metodo
        return df_top.reset_index(drop=True)

    except (IndexError, KeyError):
        # Acceder a un resultado inexistente no debe romper el flujo
        return "No hay resultados para esta busqueda"

print("Funciones de ranking definidas.")

Funciones de ranking definidas.


## 8. Ejemplos de ranking

In [109]:
# Usuario con historial
uid = int(df_reviews["user_id"].iloc[10])
ranking = recomendar(uid, model_svd, df_reviews, patrimonio, top_n=10)

if isinstance(ranking, str):
    print(ranking)
else:
    print(f"Top 10 para usuario {uid} ({ranking['metodo'].iloc[0]}):\n")
    for i, row in ranking.iterrows():
        patro = "📌 " if row.get("Patrocinado", False) else "   "
        print(f"  {i+1:>2}. {patro}{row['puntuacion_estimada']:.2f}  "
              f"{str(row['Nombre'])[:45]:<45}  [{row['Tipo de Cultura']}]")
    print()
ranking

Top 10 para usuario 294 (SVD personalizado):

   1. 📌 4.58  Palacio de John (Edificio La Bolsa)            [Patrimonio Cultural]
   2. 📌 4.55  Vivienda Minera de La Arboleda                 [Patrimonio Cultural]
   3.    4.67  Parketxe Armañón                               [Otros]
   4.    4.66  Santuario de Urkiola                           [Patrimonio Cultural]
   5.    4.64  Museo de Bellas Artes de Bilbao                [Arte]
   6.    4.62  Casa del parque Aizkorri-Aratz                 [Ciencias Naturales]
   7.    4.61  Estanque celtibérico de Laguardia              [Patrimonio Cultural]
   8.    4.59  Ermita de San Miguel de Ereñozar               [Patrimonio Cultural]
   9.    4.59  Canteras de Andrabide                          [Patrimonio Cultural]
  10.    4.58  Museo del Licor Manuel Acha                    [Otros]



,culture_id,puntuacion_estimada,Nombre,Tipo de Cultura,Tipo de lugar,Provincia,Patrocinado,metodo
0,403,4.58,Palacio de John (Edificio La Bolsa),Patrimonio Cultural,Patrimonio cultural,Bizkaia,True,SVD personalizado
1,571,4.55,Vivienda Minera de La Arboleda,Patrimonio Cultural,Patrimonio cultural,Araba,True,SVD personalizado
2,126,4.67,Parketxe Armañón,Otros,Museo,Araba,False,SVD personalizado
3,233,4.66,Santuario de Urkiola,Patrimonio Cultural,Patrimonio cultural,Araba,False,SVD personalizado
4,60,4.64,Museo de Bellas Artes de Bilbao,Arte,Museo,Bizkaia,False,SVD personalizado
5,74,4.62,Casa del parque Aizkorri-Aratz,Ciencias Naturales,Museo,Araba,False,SVD personalizado
6,543,4.61,Estanque celtibérico de Laguardia,Patrimonio Cultural,Patrimonio cultural,Araba,False,SVD personalizado
7,373,4.59,Ermita de San Miguel de Ereñozar,Patrimonio Cultural,Patrimonio cultural,Araba,False,SVD personalizado
8,336,4.59,Canteras de Andrabide,Patrimonio Cultural,Patrimonio cultural,Araba,False,SVD personalizado
9,66,4.58,Museo del Licor Manuel Acha,Otros,Museo,Araba,False,SVD personalizado


In [110]:
# Usuario nuevo (cold start)
ranking_nuevo = recomendar(99999, model_svd, df_reviews, patrimonio, top_n=100)
if isinstance(ranking_nuevo, str):
    print(ranking_nuevo)
else:
    print(f"Top 5 para usuario nuevo ({ranking_nuevo['metodo'].iloc[0]}):\n")
    for i, row in ranking_nuevo.iterrows():
        print(f"  {i+1}. {row['puntuacion_estimada']:.2f}  {row['Nombre']}")

Top 5 para usuario nuevo (Cold start (media global)):

  1. 4.73  Palacio de John (Edificio La Bolsa)
  2. 4.64  Puente de Trespuentes
  3. 4.83  La Encartada Fabrika Museoa
  4. 4.82  Parketxe Armañón
  5. 4.82  Santuario de Nuestra Señora de la Encina
  6. 4.82  Museo del Licor Manuel Acha
  7. 4.73  Museo-Territorio Lenbur
  8. 4.73  Muralla de Hondarribia
  9. 4.73  Casa del parque Aizkorri-Aratz
  10. 4.73  Estanque celtibérico de Laguardia
  11. 4.73  Ermita de San Miguel de Ereñozar
  12. 4.70  Canteras de Andrabide
  13. 4.70  Palacio Sagartegieta
  14. 4.70  Ermita Nuestra Señora de Zeberiogana
  15. 4.70  Torre-Palacio de Muntsaratz
  16. 4.70  Palacio de Olazarra-Mizkia
  17. 4.67  TOPIC. Centro Internacional del Títere de Tolosa
  18. 4.64  Santuario de Urkiola
  19. 4.64  Ekoetxea Txingudi
  20. 4.64  Coto minero de Irugurutzeta
  21. 4.64  Fuente de Berriozabaleta
  22. 4.64  Convento de la Encarnación
  23. 4.64  Iglesia de Nuestra Señora de La Asunción (Tuesta)
  24. 4.

In [111]:
# ── Filtro por provincia: acepta castellano, euskera o código ────────────────
# 'Vizcaya' == 'Bizkaia' == 48  ·  'Guipúzcoa' == 'Gipuzkoa' == 20  ·  'Álava' == 'Araba' == '01'
# 📌 = lugar patrocinado (is_sponsored=TRUE) — siempre se incluyen 2 si existen
def _mostrar(titulo, resultado):
    print(f"=== {titulo} ===")
    if isinstance(resultado, str):
        print(f"  {resultado}\n")
    else:
        for i, row in resultado.iterrows():
            patro = "📌 " if row.get("Patrocinado", False) else "   "
            print(f"  {i+1}. {patro}{row['puntuacion_estimada']:.2f}  "
                  f"{str(row['Nombre'])[:42]:<42} [{row['Tipo de lugar']}] ({row['Provincia']})")
        print()

# Misma provincia escrita de tres formas → mismo resultado
_mostrar("Top 3 en Bizkaia (euskera)",   recomendar(uid, model_svd, df_reviews, patrimonio, provincia="Bizkaia", top_n=3))
_mostrar("Top 3 en Vizcaya (castellano)", recomendar(uid, model_svd, df_reviews, patrimonio, provincia="Vizcaya", top_n=3))
_mostrar("Top 3 con código 48",           recomendar(uid, model_svd, df_reviews, patrimonio, provincia=48, top_n=3))

# Filtro por tipo de lugar (insensible a mayúsculas/tildes)
_mostrar("Top 5 Museos en Gipuzkoa",
         recomendar(uid, model_svd, df_reviews, patrimonio,
                    provincia="Gipuzkoa", tipo_lugar="museo", top_n=5))
_mostrar("Top 5 Patrimonio cultural en Araba",
         recomendar(uid, model_svd, df_reviews, patrimonio,
                    provincia="Araba", tipo_lugar="Patrimonio cultural", top_n=5))

# Provincia no reconocida → devuelve el string sin romper el flujo
_mostrar("Provincia inexistente", recomendar(uid, model_svd, df_reviews, patrimonio, provincia="Madrid"))

=== Top 3 en Bizkaia (euskera) ===
  1. 📌 4.58  Palacio de John (Edificio La Bolsa)        [Patrimonio cultural] (Bizkaia)
  2. 📌 4.39  Iglesia de San Nicolás                     [Patrimonio cultural] (Bizkaia)
  3.    4.64  Museo de Bellas Artes de Bilbao            [Museo] (Bizkaia)

=== Top 3 en Vizcaya (castellano) ===
  1. 📌 4.58  Palacio de John (Edificio La Bolsa)        [Patrimonio cultural] (Bizkaia)
  2. 📌 4.39  Iglesia de San Nicolás                     [Patrimonio cultural] (Bizkaia)
  3.    4.64  Museo de Bellas Artes de Bilbao            [Museo] (Bizkaia)

=== Top 3 con código 48 ===
  1. 📌 4.58  Palacio de John (Edificio La Bolsa)        [Patrimonio cultural] (Bizkaia)
  2. 📌 4.39  Iglesia de San Nicolás                     [Patrimonio cultural] (Bizkaia)
  3.    4.64  Museo de Bellas Artes de Bilbao            [Museo] (Bizkaia)

=== Top 5 Museos en Gipuzkoa ===
  1. 📌 4.40  Euskadiko Orkestra                         [Museo] (Gipuzkoa)
  2.    4.33  Auditorio Kursaal Pal

## 9. Guardar modelos

In [112]:
os.makedirs("ML/models", exist_ok=True)

with open("models/svd_patrimonio.pkl",       "wb") as f: pickle.dump(model_svd,     f)
with open("models/content_patrimonio.pkl",   "wb") as f: pickle.dump(pipe_content,  f)
with open("models/scaler_valoracion.pkl",    "wb") as f: pickle.dump(scaler_val,    f)

pd.Series(X_fe.columns.tolist()).to_csv("models/feature_cols_patrimonio.csv", index=False)

print("Modelos guardados en models/:")
print("  svd_patrimonio.pkl       ← modelo principal de recomendación")
print("  content_patrimonio.pkl   ← modelo de contenido (feature engineering)")
print("  scaler_valoracion.pkl    ← MinMaxScaler de valoracion")
print("  feature_cols_patrimonio  ← columnas exactas de X_fe para inferencia")

Modelos guardados en models/:
  svd_patrimonio.pkl       ← modelo principal de recomendación
  content_patrimonio.pkl   ← modelo de contenido (feature engineering)
  scaler_valoracion.pkl    ← MinMaxScaler de valoracion
  feature_cols_patrimonio  ← columnas exactas de X_fe para inferencia


## 10. Integración en el backend de Render

In [113]:
# Código para app.py / main.py en Render
# GET /get_prediction?user_id=123&top_n=10

backend_code = '''
import os, pickle, psycopg2
import pandas as pd
from flask import Flask, jsonify, request   # o FastAPI

app = Flask(__name__)

PG = dict(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "sustraiapp"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "postgres"),
)

# Cargar al arrancar
with open("ML/models/svd_patrimonio.pkl", "rb") as f:
    model_svd = pickle.load(f)

conn = psycopg2.connect(**PG, options="-c client_encoding=UTF8")
df_reviews    = pd.read_sql("SELECT user_id, culture_id, puntuacion FROM user_data.culture_reviews", conn)
df_patrimonio = pd.read_sql("""
    SELECT c.id AS culture_id, c.nombre AS "Nombre",
           c.tipo_cultura AS "Tipo de Cultura", c.tipo_lugar AS "Tipo de lugar"
    FROM market_data.culture c
""", conn)
conn.close()

@app.route("/get_prediction")
def get_prediction():
    user_id = int(request.args.get("user_id"))
    top_n   = int(request.args.get("top_n", 10))

    tiene_historial = user_id in df_reviews["user_id"].values

    if tiene_historial:
        visitados    = set(df_reviews[df_reviews["user_id"]==user_id]["culture_id"])
        no_visitados = [c for c in df_patrimonio["culture_id"] if c not in visitados]
        preds = sorted(
            [(c, model_svd.predict(user_id, c).est) for c in no_visitados],
            key=lambda x: x[1], reverse=True
        )[:top_n]
    else:
        stats = df_reviews.groupby("culture_id")["puntuacion"].mean()
        preds = sorted(stats.items(), key=lambda x: x[1], reverse=True)[:top_n]

    resultado = []
    for cid, score in preds:
        row = df_patrimonio[df_patrimonio["culture_id"]==cid].iloc[0]
        resultado.append({
            "culture_id":          int(cid),
            "nombre":              row["Nombre"],
            "tipo_cultura":        row["Tipo de Cultura"],
            "tipo_lugar":          row["Tipo de lugar"],
            "puntuacion_estimada": round(float(score), 2)
        })

    return jsonify({"user_id": user_id, "recomendaciones": resultado})
'''
print(backend_code)


import os, pickle, psycopg2
import pandas as pd
from flask import Flask, jsonify, request   # o FastAPI

app = Flask(__name__)

PG = dict(
    host=os.getenv("PGHOST", "localhost"),
    port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "sustraiapp"),
    user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD", "postgres"),
)

# Cargar al arrancar
with open("ML/models/svd_patrimonio.pkl", "rb") as f:
    model_svd = pickle.load(f)

conn = psycopg2.connect(**PG, options="-c client_encoding=UTF8")
df_reviews    = pd.read_sql("SELECT user_id, culture_id, puntuacion FROM user_data.culture_reviews", conn)
df_patrimonio = pd.read_sql("""
    SELECT c.id AS culture_id, c.nombre AS "Nombre",
           c.tipo_cultura AS "Tipo de Cultura", c.tipo_lugar AS "Tipo de lugar"
    FROM market_data.culture c
""", conn)
conn.close()

@app.route("/get_prediction")
def get_prediction():
    user_id = int(request.args.get("user_id"))
    top_n   = int(request.args.get(